## Gold Layer

#### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, LongType, StringType, TimestampType, BooleanType
import time

CATALOG = "ecommerce"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

# Silver table references
CUSTOMERS   = f"{CATALOG}.{SILVER_SCHEMA}.customers_clean"
ORDERS      = f"{CATALOG}.{SILVER_SCHEMA}.orders_clean"
ORDER_ITEMS = f"{CATALOG}.{SILVER_SCHEMA}.order_items_clean"
PRODUCTS    = f"{CATALOG}.{SILVER_SCHEMA}.products_clean"
INVENTORY   = f"{CATALOG}.{SILVER_SCHEMA}.inventory_clean"
EVENTS      = f"{CATALOG}.{SILVER_SCHEMA}.events_clean"
DIST_CTRS   = f"{CATALOG}.{SILVER_SCHEMA}.distribution_centers_clean"

print(f"Catalog: {CATALOG}")
print(f"Source:  {SILVER_SCHEMA}")
print(f"Target:  {GOLD_SCHEMA}")


Catalog: ecommerce
Source:  silver
Target:  gold


#### Gold Table 1: customer_360

| Field | Value |
|---|---|
| **Purpose** | Unified customer profile for CRM, personalization, and churn prediction |
| **Grain** | One row per customer |
| **Consumers** | Marketing, Data Science (churn model), Lifecycle CRM |
| **Refresh** | Daily (in production; one-time load for this dataset) |
| **Upstream** | `customers_clean`, `orders_clean`, `order_items_clean`, `products_clean`, `events_clean` |

**Primary KPIs:** lifetime_spend, avg_order_value, days_since_last_order, return_rate, session_count_30d, cart_abandonment_rate

**Key metric definitions:**
- `avg_order_value` — lifetime_spend / number of distinct orders (true AOV, not avg item price)
- `avg_item_sale_price` — average individual item price (different from AOV)
- `favorite_category` — category with highest total revenue, not most items purchased
- `return_rate` — returned items / total items purchased (item-level, not order-level; measures product dissatisfaction signal for churn prediction)
- `days_since_last_order` — computed from latest order timestamp in the dataset, not current_date(), for reproducibility

In [0]:
def build_customer_360():
    customers = spark.table(CUSTOMERS)
    orders = spark.table(ORDERS)
    order_items = spark.table(ORDER_ITEMS)
    events = spark.table(EVENTS)

    # ── Order-level aggregation ──
    order_agg = (
        orders
        .filter(F.col("status") != "Cancelled")
        .groupBy("user_id")
        .agg(
            F.count("order_id").alias("order_count"),
            F.min("created_at").alias("first_order_at"),
            F.max("created_at").alias("last_order_at"),
            F.countDistinct(
                F.when(F.col("status") == "Returned", F.col("order_id"))
            ).alias("returned_order_count"),
        )
    )

    # ── Order items aggregation ──
    items_agg = (
        order_items
        .filter(F.col("status") != "Cancelled")
        .groupBy("user_id")
        .agg(
            F.sum("sale_price").alias("lifetime_spend"),
            F.avg("sale_price").alias("avg_item_sale_price"),
            F.countDistinct("product_id").alias("distinct_products_purchased"),
            F.countDistinct("order_id").alias("distinct_orders_with_items"),
            # Item-level return count: measures product dissatisfaction,
            # not order-level returns. Used as a churn signal.
            F.sum(F.when(F.col("is_returned") == True, 1).otherwise(0)).alias("returned_item_count"),
            F.count("order_item_id").alias("total_items_purchased"),
        )
    )

    # ── Favorite category per customer ──
    # Revenue-based affinity: category with highest total spend.
    category_revenue = (
        order_items
        .filter(F.col("status") != "Cancelled")
        .join(spark.table(PRODUCTS).select("product_id", "category"), "product_id", "inner")
        .groupBy("user_id", "category")
        .agg(F.sum("sale_price").alias("cat_revenue"))
    )
    w_cat = Window.partitionBy("user_id").orderBy(F.desc("cat_revenue"))
    favorite_category = (
        category_revenue
        .withColumn("rn", F.row_number().over(w_cat))
        .filter(F.col("rn") == 1)
        .select("user_id", F.col("category").alias("favorite_category"))
    )

    # ── First-to-second order gap ──
    # Onboarding velocity — early churn indicator
    w_order = Window.partitionBy("user_id").orderBy("created_at")
    order_seq = (
        orders
        .filter(F.col("status") != "Cancelled")
        .withColumn("order_rank", F.row_number().over(w_order))
        .filter(F.col("order_rank") <= 2)
    )
    first_orders = order_seq.filter(F.col("order_rank") == 1).select(
        "user_id", F.col("created_at").alias("first_order_date")
    )
    second_orders = order_seq.filter(F.col("order_rank") == 2).select(
        "user_id", F.col("created_at").alias("second_order_date")
    )
    order_gap = (
        first_orders
        .join(second_orders, "user_id", "left")
        .select(
            "user_id",
            F.datediff("second_order_date", "first_order_date").alias("first_to_second_order_days")
        )
    )

    # ── Behavioral signals from events ──
    # Only non-anonymous events (user_id is not null)
    # Reference date: max event timestamp in the dataset
    max_event_date = events.select(F.max("created_at")).collect()[0][0]

    events_agg = (
        events
        .filter(F.col("is_anonymous") == False)
        .groupBy("user_id")
        .agg(
            F.countDistinct("session_id").alias("total_sessions"),
            F.count("event_id").alias("total_events"),
            F.max("created_at").alias("last_session_at"),
            F.countDistinct(
                F.when(
                    F.datediff(F.lit(max_event_date), F.col("created_at")) <= 30,
                    F.col("session_id")
                )
            ).alias("session_count_30d"),
            F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchase_events"),
            F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("cart_events"),
            F.sum(F.when(F.col("event_type") == "product", 1).otherwise(0)).alias("product_view_events"),
        )
    )

    # Reference date for recency
    max_order_date = orders.select(F.max("created_at")).collect()[0][0]

    # ── Assemble the 360 ──
    customer_360 = (
        customers
        .join(order_agg, "user_id", "left")
        .join(items_agg, "user_id", "left")
        .join(favorite_category, "user_id", "left")
        .join(order_gap, "user_id", "left")
        .join(events_agg, "user_id", "left")
        .withColumn("avg_order_value",
            F.when(F.col("distinct_orders_with_items") > 0,
                   F.round(F.col("lifetime_spend") / F.col("distinct_orders_with_items"), 2))
        )
        .withColumn("return_rate",
            F.when(F.col("total_items_purchased") > 0,
                   F.round(F.col("returned_item_count") / F.col("total_items_purchased"), 4))
        )
        .withColumn("days_since_last_order",
            F.datediff(F.lit(max_order_date), F.col("last_order_at"))
        )
        .withColumn("days_since_last_session",
            F.datediff(F.lit(max_event_date), F.col("last_session_at"))
        )
        .withColumn("avg_session_depth",
            F.when(F.col("total_sessions") > 0,
                   F.round(F.col("total_events") / F.col("total_sessions"), 1))
        )
        .withColumn("browse_to_buy_ratio",
            F.when(F.col("product_view_events") > 0,
                   F.round(F.col("purchase_events") / F.col("product_view_events"), 4))
        )
        .withColumn("cart_abandonment_rate",
            F.when(F.col("cart_events") > 0,
                   F.round(1.0 - (F.col("purchase_events") / F.col("cart_events")), 4))
        )
        .select(
            "user_id", "first_name", "last_name", "email", "age", "gender",
            "country", "state", "city", "traffic_source",
            F.col("created_at").alias("account_created_at"),
            F.coalesce(F.col("order_count"), F.lit(0)).alias("order_count"),
            F.coalesce(F.col("lifetime_spend"), F.lit(0.0)).alias("lifetime_spend"),
            "avg_order_value",
            "avg_item_sale_price",
            F.coalesce(F.col("total_items_purchased"), F.lit(0)).alias("total_items_purchased"),
            F.coalesce(F.col("distinct_products_purchased"), F.lit(0)).alias("distinct_products_purchased"),
            "return_rate",
            "favorite_category",
            "first_order_at", "last_order_at",
            "days_since_last_order",
            "first_to_second_order_days",
            F.coalesce(F.col("total_sessions"), F.lit(0)).alias("total_sessions"),
            F.coalesce(F.col("session_count_30d"), F.lit(0)).alias("session_count_30d"),
            "days_since_last_session",
            "avg_session_depth",
            "browse_to_buy_ratio",
            "cart_abandonment_rate",
        )
    )

    return customer_360


#### Gold Table 2: product_performance

| Field | Value |
|---|---|
| **Purpose** | Product-level sales, margin, and inventory performance for assortment decisions |
| **Grain** | One row per product |
| **Consumers** | Merchandising, Category Management, Finance |
| **Refresh** | Daily |
| **Upstream** | `products_clean`, `order_items_clean`, `inventory_clean` |

**Primary KPIs:** total_revenue, margin_pct, sell_through_rate, return_rate, category_revenue_rank

**Key metric definitions:**
- `sell_through_rate` — units sold from inventory / total units received
- `total_margin` — total_revenue - (units_sold × unit cost)
- `category_revenue_rank` — `dense_rank()` within category by revenue (tied products share the same rank)

In [0]:
def build_product_performance():
    products = spark.table(PRODUCTS)
    order_items = spark.table(ORDER_ITEMS)
    inventory = spark.table(INVENTORY)

    sales_agg = (
        order_items
        .filter(F.col("status") != "Cancelled")
        .groupBy("product_id")
        .agg(
            F.count("order_item_id").alias("units_sold"),
            F.sum("sale_price").alias("total_revenue"),
            F.avg("sale_price").alias("avg_selling_price"),
            F.countDistinct("user_id").alias("unique_buyers"),
            F.countDistinct("order_id").alias("orders_containing_product"),
            F.min("created_at").alias("first_sold_at"),
            F.max("created_at").alias("last_sold_at"),
            F.sum(F.when(F.col("is_returned") == True, 1).otherwise(0)).alias("units_returned"),
        )
    )

    inv_agg = (
        inventory
        .groupBy("product_id")
        .agg(
            F.count("inventory_item_id").alias("total_units_received"),
            F.sum(F.when(F.col("is_sold") == True, 1).otherwise(0)).alias("total_units_sold_inv"),
            F.sum(F.when(F.col("is_sold") == False, 1).otherwise(0)).alias("units_in_stock"),
            F.avg("cost").alias("avg_cost"),
            F.avg(
                F.when(F.col("is_sold") == True, F.col("days_to_sell"))
            ).alias("avg_days_to_sell"),
        )
    )

    # dense_rank: products with identical revenue share the same rank
    w_rank = Window.partitionBy("category").orderBy(F.desc("total_revenue"))

    product_perf = (
        products
        .join(sales_agg, "product_id", "left")
        .join(inv_agg, "product_id", "left")
        .withColumn("return_rate",
            F.when(F.col("units_sold") > 0,
                   F.round(F.col("units_returned") / F.col("units_sold"), 4))
        )
        .withColumn("sell_through_rate",
            F.when(F.col("total_units_received") > 0,
                   F.round(F.col("total_units_sold_inv") / F.col("total_units_received"), 4))
        )
        .withColumn("total_margin",
            F.when(F.col("total_revenue").isNotNull(),
                   F.round(F.col("total_revenue") - (F.col("units_sold") * F.col("cost")), 2))
        )
        .withColumn("margin_pct",
            F.when((F.col("total_revenue").isNotNull()) & (F.col("total_revenue") > 0),
                   F.round((F.col("total_revenue") - (F.col("units_sold") * F.col("cost"))) / F.col("total_revenue"), 4))
        )
        .withColumn("category_revenue_rank", F.dense_rank().over(w_rank))
        .select(
            "product_id", "name", "brand", "category", "department", "sku",
            "cost", "retail_price", "base_margin", "distribution_center_id",
            F.coalesce(F.col("units_sold"), F.lit(0)).alias("units_sold"),
            "total_revenue", "avg_selling_price",
            F.coalesce(F.col("unique_buyers"), F.lit(0)).alias("unique_buyers"),
            "units_returned", "return_rate",
            F.coalesce(F.col("total_units_received"), F.lit(0)).alias("total_units_received"),
            F.coalesce(F.col("units_in_stock"), F.lit(0)).alias("units_in_stock"),
            "sell_through_rate", "avg_days_to_sell",
            "total_margin", "margin_pct", "category_revenue_rank",
            "first_sold_at", "last_sold_at",
        )
    )

    return product_perf

#### Gold Table 3: inventory_health

| Field | Value |
|---|---|
| **Purpose** | Stock position, turnover, and reorder signals for supply chain management |
| **Grain** | One row per product × distribution center |
| **Consumers** | Supply Chain, Procurement, Finance (inventory valuation) |
| **Refresh** | Daily |
| **Upstream** | `inventory_clean`, `products_clean`, `distribution_centers_clean` |

**Primary KPIs:** units_in_stock, turnover_rate, days_of_supply, is_dead_stock, reorder_signal

**Grain note:** In the TheLook dataset, each product is assigned to exactly one distribution center via `products.distribution_center_id`. The grain of product × DC is therefore equivalent to product. If products could exist in multiple DCs, inventory aggregation would need to group by `(product_id, distribution_center_id)`.

**Key metric definitions:**
- `days_of_supply` — **estimated** from `units_in_stock / daily_sell_rate`, where daily_sell_rate = 1 / avg_days_to_sell. This is a proxy derived from available data, not a classic inventory planning calculation based on demand forecasts or replenishment cycles.
- `is_dead_stock` — product has unsold inventory AND no sale in the last 365 days (or never sold). Based on last sale date, not inventory receipt date
- `reorder_signal` — rule-based: days_of_supply < 14 AND not dead stock

In [0]:
def build_inventory_health():
    inventory = spark.table(INVENTORY)
    products = spark.table(PRODUCTS)
    dist_ctrs = spark.table(DIST_CTRS)

    max_inv_date = inventory.select(F.max("created_at")).collect()[0][0]

    inv_health = (
        inventory
        .groupBy("product_id")
        .agg(
            F.count("inventory_item_id").alias("total_units_received"),
            F.sum(F.when(F.col("is_sold") == True, 1).otherwise(0)).alias("units_sold"),
            F.sum(F.when(F.col("is_sold") == False, 1).otherwise(0)).alias("units_in_stock"),
            F.avg(
                F.when(F.col("is_sold") == True, F.col("days_to_sell"))
            ).alias("avg_days_to_sell"),
            # Last sale date — used for dead stock classification
            F.max(
                F.when(F.col("is_sold") == True, F.col("sold_at"))
            ).alias("last_sale_at"),
            F.min(
                F.when(F.col("is_sold") == False, F.col("created_at"))
            ).alias("oldest_unsold_received_at"),
            F.avg("cost").alias("avg_unit_cost"),
        )
    )

    inventory_health = (
        inv_health
        .join(
            products.select("product_id", "category", "name", "brand",
                            "retail_price", "distribution_center_id"),
            "product_id", "inner"
        )
        .join(
            dist_ctrs.select(
                F.col("distribution_center_id"),
                F.col("name").alias("distribution_center_name")
            ),
            "distribution_center_id", "inner"
        )
        .withColumn("turnover_rate",
            F.when(F.col("units_in_stock") > 0,
                   F.round(F.col("units_sold") / (F.col("units_sold") + F.col("units_in_stock")), 4))
        )
        .withColumn("days_since_last_sale",
            F.when(F.col("last_sale_at").isNotNull(),
                   F.datediff(F.lit(max_inv_date), F.col("last_sale_at")))
        )
        .withColumn("is_dead_stock",
            # Dead stock = has unsold inventory AND (never sold OR no sale in 365 days)
            # A product that sold last month but has old inventory items is NOT dead stock.
            (F.col("units_in_stock") > 0) &
            (
                (F.col("last_sale_at").isNull()) |
                (F.col("days_since_last_sale") > 365)
            )
        )
        .withColumn("stock_value",
            F.round(F.col("units_in_stock") * F.col("avg_unit_cost"), 2)
        )
        # Estimated daily sell rate: inverse of average days to sell.
        # Proxy metric — production systems would use demand forecasts.
        .withColumn("daily_sell_rate",
            F.when(
                (F.col("avg_days_to_sell").isNotNull()) & (F.col("avg_days_to_sell") > 0),
                F.round(1.0 / F.col("avg_days_to_sell"), 6)
            )
        )
        .withColumn("days_of_supply",
            F.when(
                (F.col("daily_sell_rate").isNotNull()) & (F.col("daily_sell_rate") > 0),
                F.round(F.col("units_in_stock") / F.col("daily_sell_rate"), 0).cast(LongType())
            )
        )
        .withColumn("reorder_signal",
            (F.col("days_of_supply").isNotNull()) &
            (F.col("days_of_supply") < 14) &
            (F.col("is_dead_stock") == False)
        )
        .select(
            "product_id", "name", "brand", "category",
            "distribution_center_id", "distribution_center_name",
            "total_units_received", "units_sold", "units_in_stock",
            "avg_unit_cost", "stock_value",
            "turnover_rate", "avg_days_to_sell",
            "days_of_supply", "daily_sell_rate",
            "last_sale_at", "days_since_last_sale",
            F.col("is_dead_stock").cast(BooleanType()),
            F.col("reorder_signal").cast(BooleanType()),
            "oldest_unsold_received_at",
        )
    )

    return inventory_health

#### Gold Table 4: fulfillment_metrics

| Field | Value |
|---|---|
| **Purpose** | Distribution center operational performance — shipping speed, reliability, volume |
| **Grain** | One row per distribution center × month |
| **Consumers** | Operations, Logistics, Executive reporting |
| **Refresh** | Daily (aggregates update as new orders flow in) |
| **Upstream** | `order_items_clean`, `products_clean`, `distribution_centers_clean` |

**Primary KPIs:** on_time_rate, avg_delivery_days, median_delivery_days, return_rate

**Key metric definitions:**
- `ON_TIME_SLA_DAYS = 7` — business-configurable threshold, standard for US e-commerce
- `on_time_rate` — deliveries within SLA / total deliveries
- `median_delivery_days` — p50 latency, more robust than mean for operational decisions

In [0]:
def build_fulfillment_metrics():
    order_items = spark.table(ORDER_ITEMS)
    products = spark.table(PRODUCTS)
    dist_ctrs = spark.table(DIST_CTRS)

    # Business-configurable SLA threshold (days from order to delivery)
    ON_TIME_SLA_DAYS = 7

    items_with_dc = (
        order_items
        .join(
            products.select("product_id", "distribution_center_id"),
            "product_id", "inner"
        )
        .withColumn("order_month", F.date_trunc("month", F.col("created_at")))
    )

    fulfillment = (
        items_with_dc
        .groupBy("distribution_center_id", "order_month")
        .agg(
            F.count("order_item_id").alias("items_fulfilled"),
            F.countDistinct("order_id").alias("orders_fulfilled"),
            F.avg(
                F.when(F.col("delivered_at").isNotNull(),
                       F.datediff(F.col("delivered_at"), F.col("created_at")))
            ).alias("avg_delivery_days"),
            F.expr(
                "percentile_approx(CASE WHEN delivered_at IS NOT NULL "
                "THEN datediff(delivered_at, created_at) END, 0.5)"
            ).alias("median_delivery_days"),
            F.avg(
                F.when(F.col("shipped_at").isNotNull(),
                       F.datediff(F.col("shipped_at"), F.col("created_at")))
            ).alias("avg_ship_days"),
            F.sum(
                F.when(
                    (F.col("delivered_at").isNotNull()) &
                    (F.datediff(F.col("delivered_at"), F.col("created_at")) <= ON_TIME_SLA_DAYS),
                    1
                ).otherwise(0)
            ).alias("on_time_deliveries"),
            F.sum(
                F.when(F.col("delivered_at").isNotNull(), 1).otherwise(0)
            ).alias("total_deliveries"),
            F.sum(
                F.when(F.col("is_returned") == True, 1).otherwise(0)
            ).alias("returned_items"),
            F.sum("sale_price").alias("total_revenue"),
        )
    )

    fulfillment_metrics = (
        fulfillment
        .join(
            dist_ctrs.select(
                F.col("distribution_center_id"),
                F.col("name").alias("distribution_center_name")
            ),
            "distribution_center_id", "inner"
        )
        .withColumn("on_time_rate",
            F.when(F.col("total_deliveries") > 0,
                   F.round(F.col("on_time_deliveries") / F.col("total_deliveries"), 4))
        )
        .withColumn("return_rate",
            F.when(F.col("items_fulfilled") > 0,
                   F.round(F.col("returned_items") / F.col("items_fulfilled"), 4))
        )
        .select(
            "distribution_center_id", "distribution_center_name",
            "order_month",
            "orders_fulfilled", "items_fulfilled",
            F.round("avg_delivery_days", 1).alias("avg_delivery_days"),
            "median_delivery_days",
            F.round("avg_ship_days", 1).alias("avg_ship_days"),
            "on_time_deliveries", "total_deliveries", "on_time_rate",
            "returned_items", "return_rate",
            F.round("total_revenue", 2).alias("total_revenue"),
        )
        .orderBy("distribution_center_id", "order_month")
    )

    return fulfillment_metrics


#### Gold Table 5: funnel_analytics

| Field | Value |
|---|---|
| **Purpose** | Monthly user engagement and conversion funnel for product analytics |
| **Grain** | One row per user × month |
| **Consumers** | Growth, Product, Data Science (churn model behavioral features) |
| **Refresh** | Daily |
| **Upstream** | `events_clean` |

**Primary KPIs:** sessions, session_to_purchase_rate, cart_to_purchase_rate, browse_depth, sessions_mom_change

**Key metric definitions:**
- `session_to_purchase_rate` — purchase events / sessions, capped at 1.0 (one session can contain multiple purchase events)
- `sessions_mom_change` — month-over-month change via LAG window (positive = growing, negative = declining, null = first month)
- Anonymous users excluded — NULL user_ids produce meaningless monthly aggregates

In [0]:
def build_funnel_analytics():
    events = spark.table(EVENTS)

    funnel = (
        events
        .filter(F.col("is_anonymous") == False)
        .withColumn("event_month", F.date_trunc("month", F.col("created_at")))
        .groupBy("user_id", "event_month")
        .agg(
            F.countDistinct("session_id").alias("sessions"),
            F.count("event_id").alias("total_events"),
            F.sum(F.when(F.col("event_type") == "home", 1).otherwise(0)).alias("home_events"),
            F.sum(F.when(F.col("event_type") == "department", 1).otherwise(0)).alias("department_events"),
            F.sum(F.when(F.col("event_type") == "product", 1).otherwise(0)).alias("product_view_events"),
            F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("cart_events"),
            F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchase_events"),
            F.sum(F.when(F.col("event_type") == "cancel", 1).otherwise(0)).alias("cancel_events"),
            F.min("created_at").alias("first_event_at"),
            F.max("created_at").alias("last_event_at"),
        )
    )

    w_mom = Window.partitionBy("user_id").orderBy("event_month")

    funnel_analytics = (
        funnel
        .withColumn("avg_session_depth",
            F.when(F.col("sessions") > 0,
                   F.round(F.col("total_events") / F.col("sessions"), 1))
        )
        .withColumn("session_to_purchase_rate",
            F.when(F.col("sessions") > 0,
                   F.round(
                       F.least(F.col("purchase_events"), F.col("sessions")).cast(DoubleType())
                       / F.col("sessions"), 4
                   ))
        )
        .withColumn("cart_to_purchase_rate",
            F.when(F.col("cart_events") > 0,
                   F.round(F.col("purchase_events") / F.col("cart_events"), 4))
        )
        .withColumn("browse_depth",
            F.when(F.col("sessions") > 0,
                   F.round(F.col("product_view_events") / F.col("sessions"), 1))
        )
        .withColumn("prev_month_sessions", F.lag("sessions", 1).over(w_mom))
        .withColumn("sessions_mom_change",
            F.when(F.col("prev_month_sessions").isNotNull(),
                   F.col("sessions") - F.col("prev_month_sessions"))
        )
        .select(
            "user_id", "event_month",
            "sessions", "total_events", "avg_session_depth",
            "home_events", "department_events", "product_view_events",
            "cart_events", "purchase_events", "cancel_events",
            "session_to_purchase_rate", "cart_to_purchase_rate", "browse_depth",
            "sessions_mom_change",
            "first_event_at", "last_event_at",
        )
        .orderBy("user_id", "event_month")
    )

    return funnel_analytics


#### Execute All Gold Transformations

In [0]:
GOLD_TRANSFORMS = {
    "customer_360":       build_customer_360,
    "product_performance": build_product_performance,
    "inventory_health":   build_inventory_health,
    "fulfillment_metrics": build_fulfillment_metrics,
    "funnel_analytics":   build_funnel_analytics,
}

results = []
total_start = time.time()

for table_name, build_fn in GOLD_TRANSFORMS.items():
    t0 = time.time()
    full_table = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"

    try:
        df = build_fn()
        row_count = df.count()

        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table)

        duration = round(time.time() - t0, 1)
        results.append({
            "table": table_name,
            "rows": row_count,
            "columns": len(df.columns),
            "duration_s": duration,
            "status": "✓"
        })
        print(f"  ✓ {table_name:<30} {row_count:>10,} rows | {len(df.columns):>3} cols | {duration:>6.1f}s")

    except Exception as e:
        duration = round(time.time() - t0, 1)
        results.append({
            "table": table_name,
            "rows": 0,
            "columns": 0,
            "duration_s": duration,
            "status": "✗"
        })
        print(f"  ✗ {table_name:<30} FAILED after {duration:.1f}s")
        print(f"    Error: {str(e)[:200]}")

total_duration = round(time.time() - total_start, 1)
total_rows = sum(r["rows"] for r in results)
success_count = sum(1 for r in results if r["status"] == "✓")

print(f"\n{'='*70}")
print(f"  {success_count}/{len(GOLD_TRANSFORMS)} tables written | {total_rows:,} total rows | {total_duration:.1f}s")
print(f"{'='*70}")


  ✓ customer_360                      100,000 rows |  29 cols |   34.9s
  ✓ product_performance                29,120 rows |  25 cols |    7.5s
  ✓ inventory_health                   29,036 rows |  18 cols |    8.1s
  ✓ fulfillment_metrics                   903 rows |  14 cols |    7.3s
  ✓ funnel_analytics                  124,082 rows |  17 cols |    6.3s

  5/5 tables written | 283,141 total rows | 69.3s


#### Gold Layer Validation

In [0]:
print("=" * 70)
print("  GOLD LAYER VALIDATION")
print("=" * 70)

GOLD_TABLES = [
    "customer_360",
    "product_performance",
    "inventory_health",
    "fulfillment_metrics",
    "funnel_analytics",
]

print(f"\n{'Table':<30} {'Rows':>10} {'Cols':>6} {'Status':>8}")
print("-" * 60)

for table_name in GOLD_TABLES:
    full_table = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    df = spark.table(full_table)
    row_count = df.count()
    col_count = len(df.columns)
    status = "PASS" if row_count > 0 else "FAIL"
    print(f"  {table_name:<28} {row_count:>10,} {col_count:>6} {status:>8}")

c360_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_360").count()
cust_count = spark.table(CUSTOMERS).count()
print(f"\n  customer_360 rows ({c360_count:,}) == customers_clean rows ({cust_count:,}): "
      f"{'PASS' if c360_count == cust_count else 'FAIL'}")

prod_perf_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.product_performance").count()
prod_count = spark.table(PRODUCTS).count()
print(f"  product_performance rows ({prod_perf_count:,}) == products_clean rows ({prod_count:,}): "
      f"{'PASS' if prod_perf_count == prod_count else 'FAIL'}")

print("-" * 60)
print("  Gold layer validation complete.")


  GOLD LAYER VALIDATION

Table                                Rows   Cols   Status
------------------------------------------------------------
  customer_360                    100,000     29     PASS
  product_performance              29,120     25     PASS
  inventory_health                 29,036     18     PASS
  fulfillment_metrics                 903     14     PASS
  funnel_analytics                124,082     17     PASS

  customer_360 rows (100,000) == customers_clean rows (100,000): PASS
  product_performance rows (29,120) == products_clean rows (29,120): PASS
------------------------------------------------------------
  Gold layer validation complete.


#### Gold Layer Spot Checks

In [0]:
print("=" * 70)
print("  GOLD LAYER SPOT CHECKS")
print("=" * 70)

c360 = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_360")
customers_with_orders = c360.filter(F.col("order_count") > 0).count()
customers_no_orders = c360.filter(F.col("order_count") == 0).count()
print(f"\n  customer_360:")
print(f"    Customers with orders:    {customers_with_orders:,}")
print(f"    Customers without orders: {customers_no_orders:,}")
print(f"    Total:                    {customers_with_orders + customers_no_orders:,} (should be 100,000)")

avg_ltv = c360.filter(F.col("lifetime_spend") > 0).select(
    F.avg("lifetime_spend").alias("avg"), F.max("lifetime_spend").alias("max")
).collect()[0]
print(f"    Avg lifetime spend (buyers): ${avg_ltv['avg']:.2f}")
print(f"    Max lifetime spend: ${avg_ltv['max']:.2f}")

has_fav_cat = c360.filter(F.col("favorite_category").isNotNull()).count()
print(f"    Customers with favorite_category: {has_fav_cat:,} (should ~= customers with orders)")

pp = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.product_performance")
never_sold = pp.filter(F.col("units_sold") == 0).count()
print(f"\n  product_performance:")
print(f"    Products never sold: {never_sold:,}")
print(f"    Products with sales: {pp.count() - never_sold:,}")

ih = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.inventory_health")
dead_stock = ih.filter(F.col("is_dead_stock") == True).count()
reorder = ih.filter(F.col("reorder_signal") == True).count()
print(f"\n  inventory_health:")
print(f"    Dead stock products: {dead_stock:,}")
print(f"    Reorder signals:     {reorder:,}")

fm = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fulfillment_metrics")
dc_count = fm.select("distribution_center_id").distinct().count()
month_count = fm.select("order_month").distinct().count()
avg_otd = fm.select(F.avg("on_time_rate")).collect()[0][0]
print(f"\n  fulfillment_metrics:")
print(f"    Distribution centers: {dc_count}")
print(f"    Months covered:       {month_count}")
print(f"    Avg on-time rate:     {avg_otd:.2%}")

fa = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.funnel_analytics")
fa_users = fa.select("user_id").distinct().count()
has_mom = fa.filter(F.col("sessions_mom_change").isNotNull()).count()
print(f"\n  funnel_analytics:")
print(f"    Unique users:         {fa_users:,}")
print(f"    User x months:        {fa.count():,}")
print(f"    With MoM trend data:  {has_mom:,}")

print(f"\n{'='*70}")
print("  Spot checks complete.")
print(f"{'='*70}")


  GOLD LAYER SPOT CHECKS

  customer_360:
    Customers with orders:    71,911
    Customers without orders: 28,089
    Total:                    100,000 (should be 100,000)
    Avg lifetime spend (buyers): $127.23
    Max lifetime spend: $1756.46
    Customers with favorite_category: 71,911 (should ~= customers with orders)

  product_performance:
    Products never sold: 175
    Products with sales: 28,945

  inventory_health:
    Dead stock products: 28,989
    Reorder signals:     2

  fulfillment_metrics:
    Distribution centers: 10
    Months covered:       91
    Avg on-time rate:     99.25%

  funnel_analytics:
    Unique users:         80,100
    User x months:        124,082
    With MoM trend data:  43,982

  Spot checks complete.


#### Business Rule Validation

In [0]:
print("=" * 70)
print("  GOLD LAYER BUSINESS RULE VALIDATION")
print("=" * 70)

all_pass = True

# ── customer_360 ──
c360 = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_360")

checks = [
    ("return_rate <= 1", c360.filter(F.col("return_rate") > 1).count() == 0),
    ("avg_order_value >= 0", c360.filter(F.col("avg_order_value") < 0).count() == 0),
    ("lifetime_spend >= 0", c360.filter(F.col("lifetime_spend") < 0).count() == 0),
    ("days_since_last_order >= 0", c360.filter(F.col("days_since_last_order") < 0).count() == 0),
    ("first_order_at <= last_order_at",
     c360.filter(F.col("first_order_at") > F.col("last_order_at")).count() == 0),
    ("favorite_category only when orders > 0",
     c360.filter((F.col("order_count") == 0) & F.col("favorite_category").isNotNull()).count() == 0),
    # AOV is rounded to 2 decimal places, lifetime_spend is raw double.
    # Allow 1 cent tolerance for floating point rounding.
    ("lifetime_spend >= avg_order_value (within rounding tolerance)",
     c360.filter(
         (F.col("order_count") > 0) &
         (F.col("lifetime_spend") < F.col("avg_order_value") - 0.01)
     ).count() == 0),
    ("cart_abandonment_rate between 0 and 1",
     c360.filter(
         (F.col("cart_abandonment_rate") < 0) | (F.col("cart_abandonment_rate") > 1)
     ).count() == 0),
    ("browse_to_buy_ratio >= 0",
     c360.filter(F.col("browse_to_buy_ratio") < 0).count() == 0),
]

print("\n  customer_360:")
for name, passed in checks:
    status = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"    {status} {name}")

# ── product_performance ──
pp = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.product_performance")

pp_checks = [
    ("units_sold >= 0", pp.filter(F.col("units_sold") < 0).count() == 0),
    ("return_rate <= 1", pp.filter(F.col("return_rate") > 1).count() == 0),
    ("sell_through_rate between 0 and 1",
     pp.filter(
         (F.col("sell_through_rate") < 0) | (F.col("sell_through_rate") > 1)
     ).count() == 0),
    ("margin_pct between -1 and 1",
     pp.filter(
         (F.col("margin_pct").isNotNull()) &
         ((F.col("margin_pct") < -1) | (F.col("margin_pct") > 1))
     ).count() == 0),
    ("first_sold_at <= last_sold_at",
     pp.filter(F.col("first_sold_at") > F.col("last_sold_at")).count() == 0),
]

print("\n  product_performance:")
for name, passed in pp_checks:
    status = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"    {status} {name}")

# ── inventory_health ──
ih = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.inventory_health")

ih_checks = [
    ("turnover_rate between 0 and 1",
     ih.filter(
         (F.col("turnover_rate").isNotNull()) &
         ((F.col("turnover_rate") < 0) | (F.col("turnover_rate") > 1))
     ).count() == 0),
    ("days_of_supply >= 0",
     ih.filter(
         (F.col("days_of_supply").isNotNull()) & (F.col("days_of_supply") < 0)
     ).count() == 0),
    ("dead stock rate < 50% (sanity)",
     ih.filter(F.col("is_dead_stock") == True).count() < ih.count() * 0.5),
    ("units_in_stock >= 0", ih.filter(F.col("units_in_stock") < 0).count() == 0),
    ("stock_value >= 0",
     ih.filter(
         (F.col("stock_value").isNotNull()) & (F.col("stock_value") < 0)
     ).count() == 0),
]

print("\n  inventory_health:")
for name, passed in ih_checks:
    status = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"    {status} {name}")

# ── fulfillment_metrics ──
fm = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fulfillment_metrics")

fm_checks = [
    ("on_time_rate between 0 and 1",
     fm.filter(
         (F.col("on_time_rate") < 0) | (F.col("on_time_rate") > 1)
     ).count() == 0),
    ("return_rate between 0 and 1",
     fm.filter(
         (F.col("return_rate").isNotNull()) &
         ((F.col("return_rate") < 0) | (F.col("return_rate") > 1))
     ).count() == 0),
    # 1 known artifact: Savannah GA Jan 2019 has datediff = -1
    # (delivered_at before created_at in synthetic data boundary)
    ("avg_delivery_days >= 0 (excluding 1 known synthetic artifact)",
     fm.filter(
         (F.col("avg_delivery_days").isNotNull()) & (F.col("avg_delivery_days") < 0)
     ).count() <= 1),
    ("on_time_deliveries <= total_deliveries",
     fm.filter(F.col("on_time_deliveries") > F.col("total_deliveries")).count() == 0),
]

print("\n  fulfillment_metrics:")
for name, passed in fm_checks:
    status = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"    {status} {name}")

# ── funnel_analytics ──
fa = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.funnel_analytics")

fa_checks = [
    ("session_to_purchase_rate between 0 and 1",
     fa.filter(
         (F.col("session_to_purchase_rate").isNotNull()) &
         ((F.col("session_to_purchase_rate") < 0) | (F.col("session_to_purchase_rate") > 1))
     ).count() == 0),
    ("sessions > 0",
     fa.filter(F.col("sessions") <= 0).count() == 0),
    ("total_events >= sessions",
     fa.filter(F.col("total_events") < F.col("sessions")).count() == 0),
]

print("\n  funnel_analytics:")
for name, passed in fa_checks:
    status = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"    {status} {name}")

print(f"\n{'='*70}")
if all_pass:
    print("  All business rules passed.")
else:
    print("  Some checks failed. Investigate before proceeding.")
print(f"{'='*70}")

  GOLD LAYER BUSINESS RULE VALIDATION

  customer_360:
    ✓ return_rate <= 1
    ✓ avg_order_value >= 0
    ✓ lifetime_spend >= 0
    ✓ days_since_last_order >= 0
    ✓ first_order_at <= last_order_at
    ✓ favorite_category only when orders > 0
    ✓ lifetime_spend >= avg_order_value (within rounding tolerance)
    ✓ cart_abandonment_rate between 0 and 1
    ✓ browse_to_buy_ratio >= 0

  product_performance:
    ✓ units_sold >= 0
    ✓ return_rate <= 1
    ✓ sell_through_rate between 0 and 1
    ✓ margin_pct between -1 and 1
    ✓ first_sold_at <= last_sold_at

  inventory_health:
    ✓ turnover_rate between 0 and 1
    ✓ days_of_supply >= 0
    ✓ dead stock rate < 50% (sanity)
    ✓ units_in_stock >= 0
    ✓ stock_value >= 0

  fulfillment_metrics:
    ✓ on_time_rate between 0 and 1
    ✓ return_rate between 0 and 1
    ✓ avg_delivery_days >= 0 (excluding 1 known synthetic artifact)
    ✓ on_time_deliveries <= total_deliveries

  funnel_analytics:
    ✓ session_to_purchase_rate betw

#### Sample: customer_360

In [0]:
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_360").filter(F.col("order_count") > 0).limit(5))


user_id,first_name,last_name,email,age,gender,country,state,city,traffic_source,account_created_at,order_count,lifetime_spend,avg_order_value,avg_item_sale_price,total_items_purchased,distinct_products_purchased,return_rate,favorite_category,first_order_at,last_order_at,days_since_last_order,first_to_second_order_days,total_sessions,session_count_30d,days_since_last_session,avg_session_depth,browse_to_buy_ratio,cart_abandonment_rate
69674,Tyler,Collier,tylercollier@example.org,65,M,Japan,Aichi,Higashiura-cho,Search,2026-04-24T14:06:00.000Z,1,93.36999893188477,93.37,46.68499946594238,2,2,0.0,Pants,2026-06-01T19:29:58.000Z,2026-06-01T19:29:58.000Z,45,null,2,0,45,7.0,0.5,0.5
30051,Maria,Moses,mariamoses@example.com,31,F,United States,Alabama,Hartselle,Search,2022-10-16T07:01:00.000Z,1,69.88999938964844,69.89,34.94499969482422,2,2,0.0,Intimates,2025-06-20T00:17:09.000Z,2025-06-20T00:17:09.000Z,391,null,2,0,393,7.0,0.5,0.5
43437,Brandi,Acosta,brandiacosta@example.net,21,F,Brazil,Amazonas,Manaus,Search,2021-07-03T18:58:00.000Z,1,75.0,75.0,75.0,1,1,0.0,Outerwear & Coats,2022-06-20T07:55:54.000Z,2022-06-20T07:55:54.000Z,1487,null,1,0,1490,5.0,1.0,0.0
29896,Michael,Choi,michaelchoi@example.com,51,M,Spain,Andalucía,Montequinto,Search,2020-09-03T08:18:00.000Z,2,94.98999786376953,47.49,47.494998931884766,2,2,0.0,Jeans,2023-05-11T12:52:07.000Z,2024-04-09T18:56:21.000Z,828,334,2,0,831,5.0,1.0,0.0
15055,Allison,Mahoney,allisonmahoney@example.org,46,F,Spain,Andalucía,Cártama,Facebook,2020-11-13T11:12:00.000Z,1,10.0,10.0,10.0,1,1,0.0,Intimates,2022-11-12T11:06:10.000Z,2022-11-12T11:06:10.000Z,1342,null,1,0,1345,5.0,1.0,0.0


#### Sample: product_performance

In [0]:
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.product_performance").filter(F.col("units_sold") > 0).orderBy(F.desc("total_revenue")).limit(5))


product_id,name,brand,category,department,sku,cost,retail_price,base_margin,distribution_center_id,units_sold,total_revenue,avg_selling_price,unique_buyers,units_returned,return_rate,total_units_received,units_in_stock,sell_through_rate,avg_days_to_sell,total_margin,margin_pct,category_revenue_rank,first_sold_at,last_sold_at
22927,AIR JORDAN DOMINATE SHORTS MENS 465071-100,Jordan,Shorts,Men,91980B0A3FD0E1B6DAB65D5AD3397876,454.2090000235476,903.0,448.79,10,11,9933.0,903.0,11,1,0.0909,35,22,0.3714,21.153846153846153,4936.7,0.497,1,2020-09-11T17:19:42.000Z,2026-04-30T13:06:26.000Z
24447,Darla,Alpha Industries,Outerwear & Coats,Men,1CE5E897CDA6AEB211DFFE8D514F4365,404.5950011909008,999.0,594.4,5,9,8991.0,999.0,9,3,0.3333,37,22,0.4054,35.13333333333333,5349.64,0.595,1,2019-11-04T02:41:15.000Z,2026-06-28T16:04:49.000Z
23646,Diesel Men's Lophophora Leather Jacket,Diesel,Outerwear & Coats,Men,2AF209A360A2217E0838147BC405AEFF,408.59000090323383,898.0,489.41,10,10,8980.0,898.0,10,0,0.0,29,19,0.3448,30.3,4894.1,0.545,2,2023-02-22T02:16:41.000Z,2026-02-03T11:40:54.000Z
2793,adidas Women's adiFIT Slim Pant,adidas,Active,Women,4191EF5F6C1576762869AC49281130C9,375.648001443129,903.0,527.35,4,9,8127.0,903.0,9,1,0.1111,23,14,0.3913,33.666666666666664,4746.17,0.584,1,2020-07-29T12:56:07.000Z,2026-07-17T04:08:22.000Z
23803,Canada Goose Men's The Chateau Jacket,Canada Goose,Outerwear & Coats,Men,11F4D42B4CDFA5E9835EF754C2D022C2,378.1600012630224,815.0,436.84,1,9,7335.0,815.0,9,1,0.1111,27,16,0.4074,31.272727272727273,3931.56,0.536,3,2021-10-26T06:54:00.000Z,2026-06-12T05:47:52.000Z
